In [ ]:
import sys
sys.path.append("../")

from pathlib import Path

from src.loaders.document_loader import EnterpriseDocumentLoader
from src.splitters.chunker import EnterpriseChunker
from src.embeddings.embedding_manager import EmbeddingManager

from langchain_community.vectorstores import FAISS

c:\Users\gagan\Desktop\Langchain\labs\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
loader = EnterpriseDocumentLoader()

documents = loader.load_directory("../datasets/military")

chunker = EnterpriseChunker(
    chunk_size=500,
    overlap=100
)

chunks = chunker.split_documents(documents)

print(len(chunks))

29


In [3]:
embedding_model = EmbeddingManager(
    provider="ollama",
    model_name="nomic-embed-text"
)

In [4]:
vector_db = FAISS.from_documents(
    chunks,
    embedding_model.model
)

print("Index Created")

Index Created


In [5]:
VECTOR_PATH = "../vector_store/faiss_index"

vector_db.save_local(VECTOR_PATH)

print("Saved Successfully")

Saved Successfully


In [6]:
db = FAISS.load_local(
    VECTOR_PATH,
    embedding_model.model,
    allow_dangerous_deserialization=True
)

print("Loaded")

Loaded


In [7]:
query = "Radar Calibration"

results = db.similarity_search(
    query,
    k=5
)

len(results)

5

In [8]:
for i, doc in enumerate(results):

    print("="*70)

    print("Rank :", i+1)

    print(doc.metadata)

    print()

    print(doc.page_content[:300])

Rank : 1
{'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Radar_Manual', 'source': '..\\datasets\\military\\Radar_Manual.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2', 'filename': 'Radar_Manual.pdf', 'extension': '.pdf', 'domain': 'military', 'chunk_id': 13, 'chunk_size': 226}

===
           REMEMBER:  RADAR  DISCIPLINE  SAVES  LIVES.  STAY  VIGILANT.  ========================================================================
===
Rank : 2
{'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Radar_Manual', 'source': '..\\datasets\\military\\Radar_Manual.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'filename': 'Radar_Manual.pdf', 'extension': '.pdf', 'domain': 'military', 'chunk_id': 12, 'chunk_size': 207}

in  known  enemy  territory,  then  reposition     to  ambush  counter-UAV  teams  -  Overwatch  Rotation:  Coordinate  with  squad  to  maintain  100%  radar     uptime 

In [9]:
results = db.similarity_search_with_score(
    "Radar Calibration",
    k=5
)

for doc, score in results:

    print(score)

    print(doc.metadata)

0.86367905
{'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Radar_Manual', 'source': '..\\datasets\\military\\Radar_Manual.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2', 'filename': 'Radar_Manual.pdf', 'extension': '.pdf', 'domain': 'military', 'chunk_id': 13, 'chunk_size': 226}
0.8961557
{'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Radar_Manual', 'source': '..\\datasets\\military\\Radar_Manual.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'filename': 'Radar_Manual.pdf', 'extension': '.pdf', 'domain': 'military', 'chunk_id': 12, 'chunk_size': 207}
0.9594612
{'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Radar_Manual', 'source': '..\\datasets\\military\\Radar_Manual.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'filename': 'Radar_Manual.pdf', 'extension': '.pdf', 'domain': 'military', 'chunk_id': 8, 'chun

In [10]:
retriever = db.as_retriever(
    search_kwargs={
        "k":5
    }
)

In [11]:
docs = retriever.invoke(
    "Radar Maintenance"
)

len(docs)

5

In [12]:
from vector_store.faiss_store import EnterpriseFAISS

store = EnterpriseFAISS(
    embedding_model
)

store.create(chunks)

store.save()

docs = store.search(
    "Radar Calibration",
    k=3
)

len(docs)

3

In [21]:
new_docs = loader.load_directory(
    "../datasets/military"
)

new_chunks = chunker.split_documents(
    new_docs
)

store.db.add_documents(
    new_chunks
)

store.save()